# Inference Lab: Pricing Under Uncertainty


**Run this notebook in Google Colab.** The `demand` module is compiled for Linux and Python 3.13 only, so it will not import on a Mac or Windows machine.

The goal of this lab is to use what you've learned in this class to find the optimal price for something. You can't know for sure what the underlying curve of revenue vs price looks like. All you know is what you would know in the real world, information you gain by sampling-- you can market test any price using the demand function below. When you set the price, the market test runs for a certain amount of time, and then reports responses-- a list of 1s (bought) and 0s (didn't buy) for that price point. The total number in the sample, $n$, is therefore a random variable, given by `len(responses)`. The number of consumers who buy the product is `responses.sum().` Every time you run a market test, it takes time (literally, it will take a while to run, so don't do this lab last minute!).

In [34]:
!git clone -q https://github.com/smgroves/demand_lab.git

%cd /content/demand_lab

from demand import demand
import numpy as np
import pandas as pd


responses = demand(price=14.0)

print(len(responses), responses.sum())

fatal: destination path 'demand_lab' already exists and is not an empty directory.
/content/demand_lab
197 159


## 1. Compare two prices

Run one market test with price = 30. Then test price = 25. Which price earns more revenue per potential customer?

1. For each price, estimate the proportion $\hat{p}$ of customers who buy, and the revenue per customer $\hat{r}$.
2. Estimate the standard error of each $\hat{r}$ by bootstrapping.
3. Compute a 95% confidence interval for each $\hat{r}$.
4. Compute a 95% confidence interval for the difference $\hat{r}_{25} - \hat{r}_{30}$.
   
5. Interpret the estimates, SEs, and intervals. Report which price earns more profit, and how confident you are.



In [18]:
mt1 = demand(price=30.0)
mt2 = demand(price=25.0)

# 1. Compute the sample proportion
prop_30 = mt1.mean()
prop_25 = mt2.mean()

# Compute the revenue per potential customer
r_30 = 30 * prop_30
r_25 = 25 * prop_25

In [20]:
# 2. Estimate the standard error of each r by bootstrapping
rng = np.random.default_rng(1)

boot_30 = []
boot_25 = []

for i in range(5000):
  sample_30 = rng.choice(mt1, size=len(mt1), replace=True)
  sample_25 = rng.choice(mt2, size=len(mt2), replace=True)

  boot_30.append(30*sample_30.mean())
  boot_25.append(25*sample_25.mean())

# Convert to a np array
boot_30 = np.array(boot_30)
boot_25 = np.array(boot_25)

# Compute the standard error and print (remember, the se is the sd of the sampling distribution)
se_30 = boot_30.std()
se_25 = boot_25.std()

print(f"Standard Error 1 (se_30): {se_30}")
print(f"Standard Error 2 (se_25): {se_25}")


Standard Error 1 (se_30): 0.9404960091495859
Standard Error 2 (se_25): 0.898360322932809


In [21]:
# 3. Find the 95% CIs
print(f"95% CI for r_30: ({r_30 - (1.96*se_30)},{r_30 + (1.96*se_30)})")
print(f"95% CI for r_25: ({r_25 - (1.96*se_25)},{r_25 + (1.96*se_25)})")


95% CI for r_30: (10.945472443580757,14.632216799447134)
95% CI for r_25: (7.802055297106339,11.32362776300295)


In [23]:
# 4.
diff = r_25 - r_30
se_diff = (boot_25-boot_30).std()
print(se_diff)
print(f"95% CI for r_25 - r_30: ({diff - (1.96*se_diff)},{diff + (1.96*se_diff)})")

1.3107145081829978
95% CI for r_25 - r_30: (-5.795003527497977,-0.6570026554206252)


First view the intervals for both proportions. At a price of $30, we are 95% confident that the expected revenue per customer lies within the interval ~(11,14.6). At a price of $25, we are 95% confident that the expected revenue per customer lies within the interval ~(7.8, 11.3). However, these numbers do not tell us much. If the two CIs did not overlap, we would be able to detect statistical significance. So to truly see if there is any underlying statistical difference, we take the difference between the two proportions, and get a 95% CI of ~(-5.8, -0.65). Since 0 does not reside in the interval, the difference is statistically significant. The standard errors tell us how much our estimates wobble from sample to sample. In this case, they tell us how much our proportion estimates would change if we resampled again. For example, se_30 tells us that from sample to sample, our proportion would change by ~$1 from sample to sample.

## 2. The cost of sampling

7. Run a second market test at each price. Pool the two price = 30 samples (`np.concatenate`), and pool the two price = 25 samples.
8. Recompute the SEs, the intervals, and the interval for the difference. By roughly what factor did the SE shrink? Explain why, using the SE formula.
   
9.  **Was the extra sampling worth it?** Every market test takes time, and while you're testing you aren't selling at your best price, so each test costs the company revenue. Did the first round already settle the question (its difference interval excluded 0)? If so, the second round cost time and bought nothing but a narrower interval. If not, did the second round settle it?


In [25]:
# 7.
mt3 = demand(price=30.0)
mt4 = demand(price=25.0)

# Pool together the samples
pooled_30 = np.concatenate([mt1, mt3])
pooled_25 = np.concatenate([mt2, mt4])

In [31]:
# 8.
r_30_pooled = 30 * pooled_30.mean()
r_25_pooled = 25 * pooled_25.mean()

rng = np.random.default_rng(1)

boot_30_pooled = []
boot_25_pooled = []

for i in range(5000):
    boot_30_pooled.append(30 * rng.choice(pooled_30, size=len(pooled_30), replace=True).mean())
    boot_25_pooled.append(25 * rng.choice(pooled_25, size=len(pooled_25), replace=True).mean())
boot_30_pooled = np.array(boot_30_pooled)
boot_25_pooled = np.array(boot_25_pooled)

# Get SEs
se_30_pooled = boot_30_pooled.std()
se_25_pooled = boot_25_pooled.std()

print(f"Standard Error 1 (se_30_pooled): {se_30_pooled} | Standard Error 1 (se_30): {se_30}")
print(f"Standard Error 2 (se_25_pooled): {se_25_pooled} | Standard Error 2 (se_25): {se_25}")
print()
print(f"Shrink Factor 30: {se_30/se_30_pooled}")
print(f"Shrink Factor 25: {se_25/se_25_pooled}")
print()
print(f"95% CI for r_30_pooled: ({r_30_pooled - (1.96*se_30_pooled)},{r_30_pooled + (1.96*se_30_pooled)})")
print(f"95% CI for r_25: ({r_25_pooled - (1.96*se_25_pooled)},{r_25_pooled + (1.96*se_25_pooled)})")
diff_pooled = r_25_pooled - r_30_pooled
se_diff_pooled = (boot_25_pooled-boot_30_pooled).std()
print(f"\nse_diff_pooled: {se_diff_pooled}")
print(f"95% CI for r_25_pooled - r_30_pooled: ({diff_pooled - (1.96*se_diff_pooled)},{diff_pooled + (1.96*se_diff_pooled)})")

Standard Error 1 (se_30_pooled): 0.7493980811119197 | Standard Error 1 (se_30): 0.9404960091495859
Standard Error 2 (se_25_pooled): 0.6417549552959151 | Standard Error 2 (se_25): 0.898360322932809

Shrink Factor 30: 1.2550018913233998
Shrink Factor 25: 1.3998494526911325

95% CI for r_30_pooled: (11.969436419616278,14.907076897575005)
95% CI for r_25: (7.5917178097438995,10.107397234503887)

se_diff_pooled: 0.9977724862546806
95% CI for r_25_pooled - r_30_pooled: (-6.5443332095309215,-2.633065063412574)


The SEs shrank by factors of ~1.255 and ~1.4 for 30 and 25 respectfully. In terms of the SE formula, the pooling size doubled n so SE shrank by a factor of sqrt(2). Since the standard error is the standard deviation of the sampling distribution, and the standard error shrank by that factor, the CIs for the pools also narrowed

9. No it was not worth it. Since we already knew that 30 earns more than 25 which was the original question that we were trying to answer, resampling takes more time and expends more energy, while having us arrive at the same conclusion. In regards to finding the optimal price (next question), it does give us narrower confidence intervals as SE shrinks, but it does no better at answering the original question here.

## 3. Finding the optimal price

Now find the price that maximizes revenue per customer. Search prices between about \$5 and \$100.

Every market test takes time, and time spent testing is revenue lost: while you test, you aren't selling at your best price. Your confidence intervals tell you whether another test is worth running. **Use as few market tests as you can (<25 would be good).** Your grade will be based on, in this order, whether you:
- accurately used the information you have by accurately interpreting confidence intervals
- found a good price, close to the optimal price
- and used fewer tests.

If this sounds fun to you, run with it! If you are still very confused about what to do, read the hints below.

<details><Summary>Hints</summary>
<b>Hint 1:</b> There are a few ways you could search:

- Brute force: test every price on a fine grid. This takes a market test at every price, including prices that are obviously bad. This would be a bad choice.
- Hill climbing: compare the CI for two prices, move toward the better one, and repeat. For example, start from the 25/30 comparison above; based on which price is better, choose another price \$10 away in that direction (15 if 25 was better; 40 if 30 was better). Shrink the step size as you get close to a good price. If you can't detect a difference between two prices, either shrink the step size or increase the sample size (i.e. run another market test at the same price).
- other strategies that I won't mention here.
  
<b>Hint 2:</b> You may want to try starting your tests with a few different prices across the range; there may be local maxima you will get stuck in if you just hill climb towards a single maximum.

<b>Hint 3:</b> You can make up your own strategy or try a mix of strategies. However, keep in mind that your grade will reflect how well you ran A/B testing and interpreting confidence intervals, more than how complex your strategy is. Similarly, a lucky guess with a brute force strategy would not be a good reproducible strategy.

<b>Hint 4:</b>
Turn your code from part 1 into a function, so you can easily input responses, run bootstrapping, calculate SE, and calculate confidence intervals. Then make another function where you give it two confidence intervals and make a decision based on them: sample more at the same test, or move in one direction with some step size.
   </details>

**Before you run any tests,** write down your plan: which prices you'll test first, how you'll decide where to go next, and what your intervals need to show before you stop. Then follow it, and note anywhere you changed course and why.

For any method, stop when your intervals show that another test is unlikely to change your answer by enough to matter.

**Deliverable:** A ~2-3 paragraph write up of the strategy you chose, your final recommended price, a 95% CI for $r$ at that price, the number of market tests you ran, and one or two sentences on how your intervals told you when to stop.

In [32]:
# Step 1

# Test $10, $50, $75

mt10 = demand(price=10.0)
mt50 = demand(price=50.0)
mt75 = demand(price=75.0)

In [33]:
prop_10 = mt10.mean()
prop_50 = mt50.mean()
prop_75 = mt75.mean()

r_10 = 10 * prop_10
r_50 = 50 * prop_50
r_75 = 75 * prop_75

rng = np.random.default_rng(1)

boot_10 = []
boot_50 = []
boot_75 = []

for i in range(5000):
  sample_10 = rng.choice(mt10, size=len(mt10), replace=True)
  sample_50 = rng.choice(mt50, size=len(mt50), replace=True)
  sample_75 = rng.choice(mt75, size=len(mt75), replace=True)

  boot_10.append(10*sample_10.mean())
  boot_50.append(50*sample_50.mean())
  boot_75.append(75*sample_75.mean())

boot_10 = np.array(boot_10)
boot_50 = np.array(boot_50)
boot_75 = np.array(boot_75)

se_10 = boot_10.std()
se_50 = boot_50.std()
se_75 = boot_75.std()

print(f"95% CI for r_10: ({r_10 - (1.96*se_10)},{r_10 + (1.96*se_10)})")
print(f"95% CI for r_50: ({r_50 - (1.96*se_50)},{r_50 + (1.96*se_50)})")
print(f"95% CI for r_75: ({r_75 - (1.96*se_75)},{r_75 + (1.96*se_75)})")


95% CI for r_10: (9.142755325527189,9.824986609956683)
95% CI for r_50: (12.063860808402927,19.073863742495277)
95% CI for r_75: (12.054497363120731,21.094673907597496)


In [38]:
# Note of AI Usage: This code is heavily re-useable, so I used Claude Opus 5.5 to help me recycle it for each example

diff_50_30 = r_50 - r_30_pooled
se_50_30 = (boot_50 - boot_30_pooled).std()
print(f"r_50 - r_30: diff = {diff_50_30:.2f}, 95% CI ({diff_50_30 - 1.96*se_50_30:.2f}, {diff_50_30 + 1.96*se_50_30:.2f})")

diff_75_50 = r_75 - r_50
se_75_50 = (boot_75 - boot_50).std()
print(f"r_75 - r_50: diff = {diff_75_50:.2f}, 95% CI ({diff_75_50 - 1.96*se_75_50:.2f}, {diff_75_50 + 1.96*se_75_50:.2f})")

diff_75_30 = r_75 - r_30_pooled
se_75_30 = (boot_75 - boot_30_pooled).std()
print(f"r_75 - r_30: diff = {diff_75_30:.2f}, 95% CI ({diff_75_30 - 1.96*se_75_30:.2f}, {diff_75_30 + 1.96*se_75_30:.2f})")

r_50 - r_30: diff = 2.13, 95% CI (-1.68, 5.94)
r_75 - r_50: diff = 1.01, 95% CI (-4.71, 6.72)
r_75 - r_30: diff = 3.14, 95% CI (-1.61, 7.88)


In [47]:
mt100 = demand(price=100.0)

In [48]:
prop_100 = mt100.mean()

r_100 = 100 * prop_100

rng = np.random.default_rng(2)

boot_100 = []

for i in range(5000):
    sample_100 = rng.choice(mt100, size=len(mt100), replace=True)

    boot_100.append(100*sample_100.mean())

boot_100 = np.array(boot_100)

se_100 = boot_100.std()

print(f"r_100 = {r_100}")
print(f"95% CI for r_100: ({r_100 - (1.96*se_100)},{r_100 + (1.96*se_100)})")

r_100 = 3.669724770642202
95% CI for r_100: (1.1604835166261616,6.1789660246582425)


In [49]:
diff_75_100 = r_75 - r_100
se_75_100 = (boot_75 - boot_100).std()
print(f"r_75 - r_100: diff = {diff_75_100:.2f}, 95% CI ({diff_75_100 - 1.96*se_75_100:.2f}, {diff_75_100 + 1.96*se_75_100:.2f})")

r_75 - r_100: diff = 12.90, 95% CI (7.74, 18.07)


In [50]:
mt85 = demand(price=85.0)
mt95 = demand(price=95.0)

In [51]:
prop_85 = mt85.mean()
prop_95 = mt95.mean()

r_85 = 85 * prop_85
r_95 = 95 * prop_95

rng = np.random.default_rng(5)

boot_85 = []
boot_95 = []

for i in range(5000):
  sample_85 = rng.choice(mt85, size=len(mt85), replace=True)
  sample_95 = rng.choice(mt95, size=len(mt95), replace=True)

  boot_85.append(85*sample_85.mean())
  boot_95.append(95*sample_95.mean())

boot_85 = np.array(boot_85)
boot_95 = np.array(boot_95)

se_85 = boot_85.std()
se_95 = boot_95.std()

print(f"r_85 = {r_85}, r_95 = {r_95}")
print(f"95% CI for r_85: ({r_85 - (1.96*se_85)},{r_85 + (1.96*se_85)})")
print(f"95% CI for r_95: ({r_95 - (1.96*se_95)},{r_95 + (1.96*se_95)})")

r_85 = 21.675, r_95 = 13.0
95% CI for r_85: (16.478408153233676,26.871591846766325)
95% CI for r_95: (8.443680360466356,17.556319639533644)


In [52]:
diff_85_75 = r_85 - r_75
se_85_75 = (boot_85 - boot_75).std()
print(f"r_85 - r_75: diff = {diff_85_75:.2f}, 95% CI ({diff_85_75 - 1.96*se_85_75:.2f}, {diff_85_75 + 1.96*se_85_75:.2f})")

diff_85_95 = r_85 - r_95
se_85_95 = (boot_85 - boot_95).std()
print(f"r_85 - r_95: diff = {diff_85_95:.2f}, 95% CI ({diff_85_95 - 1.96*se_85_95:.2f}, {diff_85_95 + 1.96*se_85_95:.2f})")

diff_95_100 = r_95 - r_100
se_95_100 = (boot_95 - boot_100).std()
print(f"r_95 - r_100: diff = {diff_95_100:.2f}, 95% CI ({diff_95_100 - 1.96*se_95_100:.2f}, {diff_95_100 + 1.96*se_95_100:.2f})")

r_85 - r_75: diff = 5.10, 95% CI (-1.85, 12.05)
r_85 - r_95: diff = 8.68, 95% CI (1.81, 15.54)
r_95 - r_100: diff = 9.33, 95% CI (4.08, 14.58)


In [53]:
mt80 = demand(price=80.0)

In [54]:
prop_80 = mt80.mean()
r_80 = 80 * prop_80

rng = np.random.default_rng(6)

boot_80 = []
for i in range(5000):
  sample_80 = rng.choice(mt80, size=len(mt80), replace=True)
  boot_80.append(80*sample_80.mean())
boot_80 = np.array(boot_80)

se_80 = boot_80.std()

print(f"r_80 = {r_80}")
print(f"95% CI for r_80: ({r_80 - (1.96*se_80)},{r_80 + (1.96*se_80)})")

r_80 = 24.132231404958677
95% CI for r_80: (19.514829763512836,28.74963304640452)


In [55]:
diff_80_75 = r_80 - r_75
se_80_75 = (boot_80 - boot_75).std()
print(f"r_80 - r_75: diff = {diff_80_75:.2f}, 95% CI ({diff_80_75 - 1.96*se_80_75:.2f}, {diff_80_75 + 1.96*se_80_75:.2f})")

diff_85_80 = r_85 - r_80
se_85_80 = (boot_85 - boot_80).std()
print(f"r_85 - r_80: diff = {diff_85_80:.2f}, 95% CI ({diff_85_80 - 1.96*se_85_80:.2f}, {diff_85_80 + 1.96*se_85_80:.2f})")

r_80 - r_75: diff = 7.56, 95% CI (1.14, 13.98)
r_85 - r_80: diff = -2.46, 95% CI (-9.52, 4.60)


**Plan**: Our goal is to find the price which maximizes revenue per customer. As of right now, we already know that the revenue per customer is likely better for 30 than it is for 25. This does not tell us the full story however, as we cannot assuming that this is a strict trend. There can be bumps on the graph that we need to identify.

First, I will test 10, 50, and 75 to cover the full range of prices. Since we already have revenue proportion data on 30 and 25, we do not need to run those again. Doing this, will allow us to locate the geenral neighborhood of where the price should reside. Upon looking at the 95% CIs for 10, 50, and 75, it is clear that 10 is worse as there is no overlap between its CI and the CIs of 50 and 75. We can rule 10 out and we can rule 25 out as well seeing as its difference with 30 does not contain 0 in the CI. Now we have narrowed down our range of the optimal value from 30-100. Next I looked at the differences between 75 and 50, 50 and 30 and 75 and 30, and since 0 is in all of the CIs, no conclusion can be definititevly be made, but we do notice a pattern. The midpoint of the CIs are increasing as we move up, suugesting that higher revenues come from higher prices in the range 30-75.

Next, it was time to move up. I decided to test 100 (the top of the range). Upon testing 100, the two intervals did not overlap, meaning 100 was statistically worse. Since the original range was (30, 100) and from 30-75 the revenues are rising as prices increase, as well as the fact that 100 is worse than 75, I directed my range from 75-100. I then tested two middle values, 85 and 95, and computed the differneces. Since 0 was not in the difference CI for 85-95 and 95-100, we can conclude that 95 is worse than 85, so the range gets narrowed once more from 75-85. The last test that I completed was splitting the range, by testing 80, and then testing the differences.

The final comparisons showed that 80 is significantly better than 75, since the difference CI for 80-75 of ~(1.14, 13.98) does not contain 0. However, the difference CI for 85-80 of ~(-9.52, 4.60) does contain 0, so 80 and 85 are statistically indistinguishable, with 80 having the higher estimate.

I recommend a price of 80, with an estimated revenue of roughly 24 per potential customer and a 95% CI of (19.51, 28.75). In total, I ran 7 market tests and stioped because my intervals showed that another test was unlikely to change my answer enough to matter statistically. 75 and 95 were both significantly worse than top candidates and 85 could not be distinguished from 80. Any further testing would not be worth the cost as revenues would be very similar.

**Final Note of AI Usage**: I used Claude Opus 5.5 to help me recycle code that I had already written myself, just plugging in different numbers for the market tests. The logic was all done on my own, and while I had used Claude initially to help me draft up an optimization procedure, I recognized flaws in its methods and did it myself.